# Environment Check — MMAI'26 Hackathon Setup Verification

**Run each cell in order.** Each section will tell you clearly whether your setup is ready, or what to fix if something is missing.

> If any cell shows a ❌, follow the fix instructions shown and re-run that cell before moving on. Problem holders are available to help if you get stuck.

---

## 1 — Python Version

In [1]:
import sys

major = sys.version_info.major
minor = sys.version_info.minor
patch = sys.version_info.micro
version_str = f"{major}.{minor}.{patch}"

RECOMMENDED = [(3, 11), (3, 12), (3, 13)]
MIN_VERSION  = (3, 11)
MAX_VERSION  = (3, 13)

current = (major, minor)

print(f"Python version detected: {version_str}")
print(f"Full path:               {sys.executable}")
print()

if major < 3:
    print("❌  FAIL — Python 2 is not supported. Please install Python 3.11 or later.")
elif current < MIN_VERSION:
    print(f"⚠️  WARNING — Python {version_str} is older than the recommended minimum (3.11).")
    print("   Some libraries used in this hackathon may not work correctly.")
    print("   Fix: download Python 3.11 or later from https://www.python.org/downloads/")
elif current > MAX_VERSION:
    print(f"⚠️  WARNING — Python {version_str} is newer than the tested range (3.11–3.13).")
    print("   It will likely work, but some libraries may not yet have compatible builds.")
    print("   Recommended: use Python 3.11, 3.12, or 3.13 for best compatibility.")
else:
    print(f"✅  PASS — Python {version_str} is within the recommended range (3.11–3.13).")


Python version detected: 3.12.12
Full path:               /Users/pu22650/work/MMAI26/.venv/bin/python

✅  PASS — Python 3.12.12 is within the recommended range (3.11–3.13).


## 2 — Jupyter Environment

In [2]:
import importlib.util
import importlib.metadata
import subprocess

def get_package_version(package_name):
    """Return installed version string or None."""
    spec = importlib.util.find_spec(package_name)
    if spec is None:
        return None
    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        return "installed (version unknown)"

jupyter_packages = {
    "notebook":      "Classic Jupyter Notebook",
    "jupyterlab":    "JupyterLab (modern interface)",
    "ipykernel":     "IPython kernel (required for notebooks to run)",
    "ipywidgets":    "Interactive widgets (optional but useful)",
}

all_pass = True
for pkg, label in jupyter_packages.items():
    ver = get_package_version(pkg)
    required = pkg in ("notebook", "jupyterlab", "ipykernel")
    if ver:
        print(f"✅  {label:45s} — {ver}")
    elif required:
        print(f"❌  {label:45s} — NOT FOUND")
        print(f"   Fix: pip install {pkg}")
        all_pass = False
    else:
        print(f"ℹ️   {label:45s} — not installed (optional)")

print()
if all_pass:
    print("✅  PASS — Jupyter environment is ready.")
else:
    print("❌  FAIL — One or more required Jupyter packages are missing. See fixes above.")
    print("   Quick fix for all at once: pip install notebook jupyterlab ipykernel")


✅  Classic Jupyter Notebook                      — 7.5.7
✅  JupyterLab (modern interface)                 — 4.5.8
✅  IPython kernel (required for notebooks to run) — 7.2.0
ℹ️   Interactive widgets (optional but useful)     — not installed (optional)

✅  PASS — Jupyter environment is ready.


## 3 — Core Data Science Libraries

In [3]:
from packaging.version import Version

# (package_import_name, pip_install_name, min_version, required)
LIBRARIES = [
    ("numpy",       "numpy",          "1.24.0",  True),
    ("pandas",      "pandas",         "2.0.0",   True),
    ("matplotlib",  "matplotlib",     "3.7.0",   True),
    ("seaborn",     "seaborn",        "0.12.0",  True),
    ("sklearn",     "scikit-learn",   "1.3.0",   True),
    ("scipy",       "scipy",          "1.11.0",  True),
    ("plotly",      "plotly",         "5.0.0",   False),
    ("shap",        "shap",           "0.42.0",  False),
    ("xgboost",     "xgboost",        "1.7.0",   False),
]

all_required_pass = True

for import_name, pip_name, min_ver, required in LIBRARIES:
    ver = get_package_version(import_name)
    tag = "required" if required else "optional"
    if ver is None:
        if required:
            print(f"❌  [{tag}] {pip_name:20s} — NOT INSTALLED   →  pip install {pip_name}")
            all_required_pass = False
        else:
            print(f"ℹ️   [{tag}] {pip_name:20s} — not installed  (install if your challenge needs it)")
    else:
        try:
            if Version(ver) < Version(min_ver):
                status = f"⚠️  [{tag}]"
                note   = f"version {ver} is below recommended {min_ver}  →  pip install --upgrade {pip_name}"
                print(f"{status} {pip_name:20s} — {note}")
                if required:
                    all_required_pass = False
            else:
                print(f"✅  [{tag}] {pip_name:20s} — {ver}")
        except Exception:
            print(f"✅  [{tag}] {pip_name:20s} — {ver} (could not compare versions)")

print()
if all_required_pass:
    print("✅  PASS — All required libraries are installed and meet minimum versions.")
else:
    print("❌  FAIL — Some required libraries are missing or outdated. See fixes above.")
    print("   Quick install: pip install numpy pandas matplotlib seaborn scikit-learn scipy")


✅  [required] numpy                — 2.4.6
✅  [required] pandas               — 3.0.3
✅  [required] matplotlib           — 3.10.9
✅  [required] seaborn              — 0.13.2
✅  [required] scikit-learn         — installed (version unknown) (could not compare versions)
✅  [required] scipy                — 1.17.1
ℹ️   [optional] plotly               — not installed  (install if your challenge needs it)
✅  [optional] shap                 — 0.52.0
✅  [optional] xgboost              — 3.2.0

✅  PASS — All required libraries are installed and meet minimum versions.


## 4 — IDE Detection

In [4]:
import os
import shutil

print("Detecting your environment and IDE...\n")

# --- Detect notebook interface ---
detected_env = "Unknown"

try:
    shell = get_ipython().__class__.__name__
    if "ZMQInteractiveShell" in shell:
        # Running inside Jupyter — check which flavour
        server_url = ""
        try:
            import notebook
            detected_env = f"Jupyter Notebook (classic)  v{notebook.__version__}"
        except Exception:
            pass
        try:
            import jupyterlab
            detected_env = f"JupyterLab  v{jupyterlab.__version__}"
        except Exception:
            pass
        # VS Code sets this env var when running notebooks
        if os.environ.get("VSCODE_PID") or os.environ.get("VSCODE_CWD"):
            detected_env = "VS Code — Jupyter notebook extension"
        if os.environ.get("PYCHARM_HOSTED"):
            detected_env = "PyCharm — Jupyter notebook"
    elif shell == "TerminalInteractiveShell":
        detected_env = "IPython terminal (not a notebook)"
except NameError:
    detected_env = "Plain Python script (not a notebook)"

print(f"  Current environment : {detected_env}")

# --- Detect IDEs available on PATH ---
IDE_CHECKS = [
    ("code",         "Visual Studio Code"),
    ("code-insiders","VS Code Insiders"),
    ("pycharm",      "PyCharm"),
    ("pycharm64",    "PyCharm (64-bit Windows)"),
    ("spyder",       "Spyder"),
    ("thonny",       "Thonny"),
    ("idle",         "IDLE (built-in Python IDE)"),
]

print("\nIDEs found on your system PATH:")
found_any = False
for cmd, label in IDE_CHECKS:
    if shutil.which(cmd):
        print(f"  ✅  {label}")
        found_any = True

if not found_any:
    print("  ℹ️   No IDEs detected on PATH.")
    print("      This does not mean you have no IDE — some (e.g. PyCharm, Anaconda)")
    print("      do not always register on the system PATH.")
    print("      Recommended IDEs if you need to install one:")
    print("        • VS Code      → https://code.visualstudio.com/")
    print("        • PyCharm CE   → https://www.jetbrains.com/pycharm/download/")
    print("        • Spyder       → pip install spyder  (or via Anaconda)")

print()
print("ℹ️   If you are running this notebook inside VS Code or PyCharm,")
print("    those IDEs are confirmed as your environment above — you are good to go.")


Detecting your environment and IDE...

  Current environment : VS Code — Jupyter notebook extension

IDEs found on your system PATH:
  ✅  Visual Studio Code
  ✅  Spyder

ℹ️   If you are running this notebook inside VS Code or PyCharm,
    those IDEs are confirmed as your environment above — you are good to go.


## 5 — Quick Smoke Test (does everything actually work?)

In [5]:
import traceback

results = {}

# numpy
try:
    import numpy as np
    arr = np.array([1.0, 2.0, 3.0])
    assert arr.mean() == 2.0
    results["numpy"] = ("✅", f"array mean = {arr.mean()}")
except Exception as e:
    results["numpy"] = ("❌", str(e))

# pandas
try:
    import pandas as pd
    df = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})
    assert len(df) == 3
    results["pandas"] = ("✅", f"DataFrame shape = {df.shape}")
except Exception as e:
    results["pandas"] = ("❌", str(e))

# matplotlib (non-interactive render)
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots()
    ax.plot([1, 2, 3], [4, 5, 6])
    plt.close(fig)
    results["matplotlib"] = ("✅", "plot created and closed cleanly")
except Exception as e:
    results["matplotlib"] = ("❌", str(e))

# seaborn
try:
    import seaborn as sns
    import pandas as pd, numpy as np
    tip = pd.DataFrame({"x": np.random.rand(20), "y": np.random.rand(20)})
    fig, ax = plt.subplots()
    sns.scatterplot(data=tip, x="x", y="y", ax=ax)
    plt.close(fig)
    results["seaborn"] = ("✅", "scatter plot created cleanly")
except Exception as e:
    results["seaborn"] = ("❌", str(e))

# scikit-learn
try:
    from sklearn.linear_model import LogisticRegression
    import numpy as np
    X = np.array([[0], [1], [2], [3]])
    y = np.array([0, 0, 1, 1])
    model = LogisticRegression().fit(X, y)
    preds = model.predict(X)
    results["scikit-learn"] = ("✅", f"predictions = {preds.tolist()}")
except Exception as e:
    results["scikit-learn"] = ("❌", str(e))

# scipy
try:
    from scipy import stats
    t_stat, p_val = stats.ttest_1samp([1, 2, 3, 4, 5], 3)
    results["scipy"] = ("✅", f"t-test ran, p = {p_val:.3f}")
except Exception as e:
    results["scipy"] = ("❌", str(e))

# json (stdlib — always should pass)
try:
    import json
    data = json.dumps({"status": "ok", "value": 42})
    parsed = json.loads(data)
    assert parsed["value"] == 42
    results["json (stdlib)"] = ("✅", "serialise/deserialise OK")
except Exception as e:
    results["json (stdlib)"] = ("❌", str(e))

# Print results
print("Smoke test results:\n")
all_pass = True
for lib, (icon, note) in results.items():
    print(f"  {icon}  {lib:20s} — {note}")
    if icon == "❌":
        all_pass = False

print()
if all_pass:
    print("✅  PASS — All smoke tests completed successfully. Your environment is ready.")
else:
    print("❌  FAIL — One or more smoke tests failed. See messages above.")
    print("   Try: pip install --upgrade <package-name>")
    print("   Then restart the kernel and re-run this cell.")


Smoke test results:

  ✅  numpy                — array mean = 2.0
  ✅  pandas               — DataFrame shape = (3, 2)
  ✅  matplotlib           — plot created and closed cleanly
  ✅  seaborn              — scatter plot created cleanly
  ✅  scikit-learn         — predictions = [0, 0, 1, 1]
  ✅  scipy                — t-test ran, p = 1.000
  ✅  json (stdlib)        — serialise/deserialise OK

✅  PASS — All smoke tests completed successfully. Your environment is ready.


## 6 — Final Summary

In [6]:
print("=" * 60)
print("  ENVIRONMENT CHECK SUMMARY")
print("=" * 60)
print()
print(f"  Python version   : {sys.version.split()[0]}")
print(f"  Python path      : {sys.executable}")
print(f"  Platform         : {sys.platform}")
print()

summary_checks = {
    "Python 3.11–3.13"           : (3, 11) <= (sys.version_info.major, sys.version_info.minor) <= (3, 13),
    "Jupyter / ipykernel"        : get_package_version("ipykernel") is not None,
    "Core libraries installed"   : all(get_package_version(p) is not None for p in ["numpy","pandas","matplotlib","seaborn","sklearn","scipy"]),
    "Smoke tests passed"         : all_pass,
}

all_green = True
for check, passed in summary_checks.items():
    icon = "✅" if passed else "❌"
    print(f"  {icon}  {check}")
    if not passed:
        all_green = False

print()
if all_green:
    print("  🎉  Everything looks good — you are ready for the hackathon!")
else:
    print("  ⚠️   One or more checks did not pass.")
    print("      Scroll back through the sections above for specific fix instructions.")
    print("      Ask a problem holder or the technical support desk if you are stuck.")
print()
print("=" * 60)


  ENVIRONMENT CHECK SUMMARY

  Python version   : 3.12.12
  Python path      : /Users/pu22650/work/MMAI26/.venv/bin/python
  Platform         : darwin

  ✅  Python 3.11–3.13
  ✅  Jupyter / ipykernel
  ✅  Core libraries installed
  ✅  Smoke tests passed

  🎉  Everything looks good — you are ready for the hackathon!

